### 下載檔案

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("joebeachcapital/30000-spotify-songs")

print("Path to dataset files:", path)

Path to dataset files: /Users/angus3938/.cache/kagglehub/datasets/joebeachcapital/30000-spotify-songs/versions/2


In [3]:
import pandas as pd

view = pd.read_csv(f"{path}/spotify_songs.csv")  # 根據實際檔名修改

---

## Testing 

---

# version 1 
### 轉換時還是用到pandas中pd.dataframe

In [41]:
def _attempt_convert_type(value_str):
    try:
        return int(value_str)
    except ValueError:
        try:
            return float(value_str)
        except ValueError:
            return value_str.strip()

def load_csv(filePath, separator=","):
    """
    utilize 'with open' to read data row by row 
    """
    data_dict = {}
    header = []

    try:
        with open(filePath, "r", encoding="utf-8") as f:
            header_line = f.readline().strip()
            header = header_line.split(separator)
            data_dict = {col_name: [] for col_name in header}

            for line in f:
                cleaned_line = line.strip()

                if not cleaned_line:
                    continue

                values = cleaned_line.split(separator)

                if len(values) == len(header):
                    for i, col_name in enumerate(header):
    
                        converted_value = _attempt_convert_type(values[i]) 
                        data_dict[col_name].append(converted_value)

                # elif len(values) > 1:
                #     print(f"Warning: Skipping line due to mismatch in column count: {cleaned_line}")

    
        if data_dict:
            return pd.DataFrame(data_dict)
        else:
            print("File loaded but no data rows were found.")
    
    except FileNotFoundError:
        print(f"File doesn't exist at {filePath}")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [42]:
data = load_csv(f"{path}/spotify_songs.csv", separator=separator)

In [43]:
print(data.head())

                 track_id                                         track_name  \
0  6f807x0ima9a1j3VPbc7VN  I Don't Care (with Justin Bieber) - Loud Luxur...   
1  0r7CVbZTWZgbTCYdfa2P31                    Memories - Dillon Francis Remix   
2  1z1Hg7Vb0AhHDiEmnDE79l                    All the Time - Don Diablo Remix   
3  75FpbthrwQmzHlBJLuGdC7                  Call You Mine - Keanu Silva Remix   
4  1e8PAfcKUYoKkxPhrHqw4x            Someone You Loved - Future Humans Remix   

       track_artist  track_popularity          track_album_id  \
0        Ed Sheeran                66  2oCs0DGTsRO98Gh5ZSl2Cx   
1          Maroon 5                67  63rPSO264uRjW1X5E6cWv6   
2      Zara Larsson                70  1HoSmj2eLcsrR0vE9gThr4   
3  The Chainsmokers                60  1nqYsOef1yKKuGOVchbsk6   
4     Lewis Capaldi                69  7m7vv9wlQ4i0LFuJiE2zsQ   

                                    track_album_name track_album_release_date  \
0  I Don't Care (with Justin Bieber) [Loud Luxu

---

# version 2 
### advancing CSV loader
### define own Dataframe

In [ ]:
def _attempt_convert_type(value_str):
    """Automatically convert data types"""
    # Handle empty strings
    if not value_str or value_str.strip() == '':
        return None
    
    value_str = value_str.strip()
    
    # Handle quoted strings
    if value_str.startswith('"') and value_str.endswith('"'):
        return value_str[1:-1]
    
    # Try converting to integer
    try:
        return int(value_str)
    except ValueError:
        pass
    
    # Try converting to float
    try:
        return float(value_str)
    except ValueError:
        pass
    
    # Handle boolean values
    if value_str.lower() in ['true', 'false']:
        return value_str.lower() == 'true'
    
    # Return string
    return value_str


def load_csv_advanced(filePath, separator=",", quote_char='"'):
    """
    advancing CSV loader, adressing quoted fields in CSV
    (ex: "Smith, John",25,USA)
    """
    def parse_csv_line(line, separator=",", quote_char='"'):
        """Parse a CSV line, correctly handling quoted fields"""
        values = []
        current_value = ""
        in_quotes = False
        
        for char in line:
            if char == quote_char:
                in_quotes = not in_quotes
            elif char == separator and not in_quotes:
                values.append(current_value)
                current_value = ""
            else:
                current_value += char
        
        # Add the last value
        values.append(current_value)
        return values
    
    data_dict = {}
    header = []
    
    try:
        with open(filePath, "r", encoding="utf-8") as f:
            # Read header line
            header_line = f.readline().strip()
            header = parse_csv_line(header_line, separator, quote_char)
            header = [col.strip() for col in header]
            
            data_dict = {col_name: [] for col_name in header}
            
            # Read data lines
            for line_number, line in enumerate(f, start=2):
                cleaned_line = line.strip()
                if not cleaned_line:
                    continue
                
                values = parse_csv_line(cleaned_line, separator, quote_char)
                
                if len(values) == len(header):
                    for i, col_name in enumerate(header):
                        converted_value = _attempt_convert_type(values[i])
                        data_dict[col_name].append(converted_value)
                else:
                    print(f"Warning: Line {line_number} column mismatch")
        
        return data_dict
        
    except Exception as e:
        print(f"Error: {e}")
        return None
    
# 

# ==================== CSV Parser with Chunk Support ====================
# NOTE: removed duplicate definition of _attempt_convert_type to avoid shadowing.
# The first definition (earlier in this cell) will be used for type conversion.

def parse_csv_line(line, separator=",", quote_char='"'):
    """解析一行 CSV"""
    values = []
    current_value = ""
    in_quotes = False
    
    for char in line:
        if char == quote_char:
            in_quotes = not in_quotes
        elif char == separator and not in_quotes:
            values.append(current_value)
            current_value = ""
        else:
            current_value += char
    
    values.append(current_value)
    return values

# ==================== Chunked CSV Reader (Generator) ====================

class CSVChunkReader:
    """
    分塊讀取 CSV 文件
    使用生成器模式,每次返回指定數量的行
    適用於處理大型 CSV 文件
    """
    
    def __init__(self, filepath, separator=",", chunk_size=1000):
        """
        初始化分塊讀取器
        
        參數:
            filepath: str - CSV 文件路徑
            separator: str - 分隔符
            chunk_size: int - 每塊的行數
        """
        self.filepath = filepath
        self.separator = separator
        self.chunk_size = chunk_size
        self.header = None
        self.total_rows_read = 0
    
    def __iter__(self):
        """返迭代器"""
        return self.read_chunks()
    
    def read_chunks(self):
        """
        生成器函數,逐塊返回 DataFrame
        
        Yields:
            DataFrame - 每次返回 chunk_size 行的 DataFrame
        """
        try:
            with open(self.filepath, 'r', encoding='utf-8') as f:
                # 讀取標題行
                header_line = f.readline().strip()
                if not header_line:
                    raise ValueError("File is empty")
                
                self.header = parse_csv_line(header_line, self.separator)
                self.header = [col.strip() for col in self.header]
                
                chunk_data = {col: [] for col in self.header}
                chunk_row_count = 0
                
                # 逐行讀取
                for line_number, line in enumerate(f, start=2):
                    cleaned_line = line.strip()
                    
                    if not cleaned_line:
                        continue
                    
                    values = parse_csv_line(cleaned_line, self.separator)
                    
                    if len(values) == len(self.header):
                        # 添加到當前塊
                        for i, col in enumerate(self.header):
                            converted_value = _attempt_convert_type(values[i])
                            chunk_data[col].append(converted_value)
                        
                        chunk_row_count += 1
                        self.total_rows_read += 1
                        
                        # 如果達到塊大小,返回這個塊
                        if chunk_row_count >= self.chunk_size:
                            yield DataFrame(chunk_data)
                            # 重置塊數據
                            chunk_data = {col: [] for col in self.header}
                            chunk_row_count = 0
                    else:
                        print(f"Warning: Line {line_number} column mismatch")
                
                # 返回最後一個不完整的塊
                if chunk_row_count > 0:
                    yield DataFrame(chunk_data)
        
        except FileNotFoundError:
            print(f"Error: File not found at {self.filepath}")
            return
        except Exception as e:
            print(f"Error reading CSV: {e}")
            import traceback
            traceback.print_exc()
            return


def read_csv_chunks(filepath, separator=",", chunk_size=1000):
    """
    便捷函數:分塊讀取 CSV
    
    參數:
        filepath: str - CSV 文件路徑
        separator: str - 分隔符
        chunk_size: int - 每塊的行數
    
    Returns:
        CSVChunkReader - 迭代器
    
    示例:
        for chunk in read_csv_chunks("large_file.csv", chunk_size=1000):
            # 處理每個塊
            print(chunk.shape())
    """
    return CSVChunkReader(filepath, separator, chunk_size)


# ==================== Chunk Processing Utilities ====================

class ChunkProcessor:
    """
    分塊處理器 - 提供常用的分塊處理操作
    """
    
    @staticmethod
    def filter_chunks(chunk_reader, condition):
        """
        對每個塊應用過濾
        
        參數:
            chunk_reader: CSVChunkReader
            condition: callable - 過濾條件
        
        Yields:
            DataFrame - 過濾後的塊
        """
        for chunk in chunk_reader:
            filtered = chunk.filter(condition)
            if len(filtered) > 0:
                yield filtered
    
    @staticmethod
    def aggregate_chunks(chunk_reader, group_by_cols, agg_dict):
        """
        對所有塊進行聚合
        
        參數:
            chunk_reader: CSVChunkReader
            group_by_cols: str or list - 分組列
            agg_dict: dict - 聚合操作
        
        Returns:
            DataFrame - 聚合結果
        """
        if isinstance(group_by_cols, str):
            group_by_cols = [group_by_cols]
        
        # 累積的分組數據
        accumulated_groups = {}
        
        for chunk in chunk_reader:
            # 對當前塊進行分組
            grouped = chunk.groupby(group_by_cols)
            chunk_result = grouped.aggregate(agg_dict)
            
            # 合併到累積結果
            for i in range(len(chunk_result)):
                # 創建組鍵
                group_key = tuple(chunk_result.data[col][i] for col in group_by_cols)
                
                if group_key not in accumulated_groups:
                    accumulated_groups[group_key] = {}
                    for col in group_by_cols:
                        accumulated_groups[group_key][col] = chunk_result.data[col][i]
                    for agg_col in agg_dict.keys():
                        for func in [agg_dict[agg_col]]:
                            result_col = f"{agg_col}_{func}"
                            accumulated_groups[group_key][result_col] = []
                
                # 累積聚合值
                for agg_col in agg_dict.keys():
                    func = agg_dict[agg_col]
                    result_col = f"{agg_col}_{func}"
                    accumulated_groups[group_key][result_col].append(
                        chunk_result.data[result_col][i]
                    )
        
        # 最終聚合
        final_data = {col: [] for col in group_by_cols}
        for agg_col in agg_dict.keys():
            func = agg_dict[agg_col]
            final_data[f"{agg_col}_{func}"] = []
        
        for group_key, group_data in accumulated_groups.items():
            # 添加分組鍵
            for i, col in enumerate(group_by_cols):
                final_data[col].append(group_key[i])
            
            # 最終聚合計算
            for agg_col in agg_dict.keys():
                func = agg_dict[agg_col]
                result_col = f"{agg_col}_{func}"
                values = group_data[result_col]
                
                if func == 'sum':
                    final_value = sum(values)
                elif func == 'max':
                    final_value = max(values)
                elif func == 'min':
                    final_value = min(values)
                elif func == 'count':
                    final_value = sum(values)
                elif func == 'mean':
                    # 需要重新計算平均值
                    final_value = sum(values) / len(values)
                else:
                    final_value = values[0]
                
                final_data[result_col].append(final_value)
        
        return DataFrame(final_data)
    
    @staticmethod
    def count_chunks(chunk_reader):
        """
        計算所有塊的總行數
        
        參數:
            chunk_reader: CSVChunkReader
        
        Returns:
            int - 總行數
        """
        total = 0
        for chunk in chunk_reader:
            total += len(chunk)
        return total
    
    @staticmethod
    def collect_chunks(chunk_reader, max_rows=None):
        """
        將所有塊合併成一個 DataFrame
        注意: 只適用於能裝入內存的情況
        
        參數:
            chunk_reader: CSVChunkReader
            max_rows: int - 最多收集多少行
        
        Returns:
            DataFrame - 合併後的 DataFrame
        """
        all_data = None
        total_rows = 0
        
        for chunk in chunk_reader:
            if all_data is None:
                all_data = {col: [] for col in chunk.columns}
            
            for col in chunk.columns:
                all_data[col].extend(chunk.data[col])
            
            total_rows += len(chunk)
            
            if max_rows and total_rows >= max_rows:
                # 截斷到 max_rows
                for col in all_data.keys():
                    all_data[col] = all_data[col][:max_rows]
                break
        
        if all_data:
            return DataFrame(all_data)
        return None



# ==================== DataFrame Class ====================

class DataFrame:
    def __init__(self, data_dict):
        """
        initialize DataFrame with a dictionary of columns
        data_dict: {'column1': [values], 'column2': [values]}
        """
        if not data_dict:
            raise ValueError("Cannot create DataFrame from empty data")
        
        self.data = data_dict
        self.columns = list(data_dict.keys())
        
        # Check that all columns have the same length
        lengths = [len(v) for v in data_dict.values()]
        if len(set(lengths)) > 1:
            raise ValueError("All columns must have the same length")
        
        self.row_count = lengths[0] if lengths else 0
    
    def __repr__(self):
        """Print DataFrame"""
        if self.row_count == 0:
            return "Empty DataFrame"
        
        # calculate column widths based on header and first 10 rows
        col_widths = {}
        for col in self.columns:
            max_width = len(col)
            for val in self.data[col][:10]:  # only check first 10 rows
                max_width = max(max_width, len(str(val)))
            col_widths[col] = min(max_width, 20)  # max 20 characters
        
        # build table
        result = []
        
        # header row
        header = " | ".join(col.ljust(col_widths[col]) for col in self.columns)
        result.append(header)
        result.append("-" * len(header))
        
        # data rows (only show first 10 rows)
        display_rows = min(10, self.row_count)
        for i in range(display_rows):
            row = []
            for col in self.columns:
                val = str(self.data[col][i])
                if len(val) > col_widths[col]:
                    val = val[:col_widths[col]-3] + "..."
                row.append(val.ljust(col_widths[col]))
            result.append(" | ".join(row))
        
        if self.row_count > 10:
            result.append(f"\n... {self.row_count - 10} more rows")
        
        result.append(f"\nShape: ({self.row_count} rows, {len(self.columns)} columns)")
        
        return "\n".join(result)
    
    def __getitem__(self, key):
        """
        supporting df['column'] and df[condition]
        """
        if isinstance(key, str):
            # return single column
            return self.data[key]
        elif isinstance(key, list) and all(isinstance(k, str) for k in key):
            # return multiple columns
            new_data = {col: self.data[col] for col in key}
            return DataFrame(new_data)
        elif isinstance(key, list) and all(isinstance(k, bool) for k in key):
            # boolean indexing (for filtering)
            if len(key) != self.row_count:
                raise ValueError("Boolean index length mismatch")
            
            new_data = {col: [] for col in self.columns}
            for i, keep in enumerate(key):
                if keep:
                    for col in self.columns:
                        new_data[col].append(self.data[col][i])
            
            return DataFrame(new_data)
        else:
            raise TypeError(f"Invalid indexing type: {type(key)}")
    
    def __len__(self):
        """return"""
        return self.row_count

# ==================== 1. Filtering ====================
    
    def filter(self, condition):
        """
        based on condition filter rows
        
        Parameters:
            condition: function or list of booleans
                - function: lambda row: row['Age'] > 18
                - list of booleans: [True, False, True, ...]
        
        Returns:
            DataFrame: filtered new DataFrame
        
        Examples:
            df.filter(lambda row: row['GNP'] > 100000)
            df.filter([True, False, True])
        """
        if callable(condition):
            # function condition
            keep_rows = []
            for i in range(self.row_count):
                row = {col: self.data[col][i] for col in self.columns}
                keep_rows.append(condition(row))
            return self[keep_rows]
        
        elif isinstance(condition, list) and all(isinstance(k, bool) for k in condition):
            # list of booleans
            return self[condition]
        
        else:
            raise TypeError("Condition must be a callable or list of booleans")
    
    # ==================== 2. Projection (Select) ====================
    
    def select(self, columns):
        """
        select specific columns from the DataFrame
        
        Parameters:
            columns: str or list
                - single column: 'Name'
                - multiple columns: ['Name', 'Age']
        
        Returns:
            DataFrame: new DataFrame containing only the selected columns
        
        Examples:
            df.select('Name')
            df.select(['Name', 'Age'])
        """
        if isinstance(columns, str):
            columns = [columns]
        
        return self[columns]
    
    # ==================== 3. GroupBy ====================
    
    def groupby(self, by):
        """
        group by one or more columns
        
        Parameters:
            by: str or list - column name(s) to group by
        
        Returns:
            GroupBy: GroupBy object
        
        Examples:
            df.groupby('Continent')
            df.groupby(['Continent', 'Country'])
        """
        if isinstance(by, str):
            by = [by]
        
        return GroupBy(self, by)
    
    # ==================== 4. Join ====================
    
    def join(self, other, left_on, right_on, how='inner'):
        """
        Join with another DataFrame
        
        Parameters:
            other: DataFrame - the other DataFrame to join with
            left_on: str - join key from the left DataFrame
            right_on: str - join key from the right DataFrame
            how: str - join type ('inner', 'left', 'right', 'outer')
        
        Returns:
            DataFrame: new DataFrame after join
        
        Examples:
            countries.join(languages, left_on='Code', right_on='CountryCode')
        """
        # Create an index for the right DataFrame based on the join key
        right_index = {}
        for i, val in enumerate(other.data[right_on]):
            if val not in right_index:
                right_index[val] = []
            right_index[val].append(i)
        
        # Initialize result dictionary
        result_data = {col: [] for col in self.columns}
        for col in other.columns:
            if col != right_on:  # Avoid duplicates
                result_data[col] = []
        
        matched_right_indices = set()
        
        # Iterate over the left DataFrame
        for i in range(self.row_count):
            left_key = self.data[left_on][i]
            
            if left_key in right_index:
                # Found matches
                for right_i in right_index[left_key]:
                    matched_right_indices.add(right_i)
                    
                    # Add left DataFrame data
                    for col in self.columns:
                        result_data[col].append(self.data[col][i])
                    
                    # Add right DataFrame data
                    for col in other.columns:
                        if col != right_on:
                            result_data[col].append(other.data[col][right_i])
            elif how in ['left', 'outer']:
                # Left join or full outer join: keep left table rows
                for col in self.columns:
                    result_data[col].append(self.data[col][i])
                for col in other.columns:
                    if col != right_on:
                        result_data[col].append(None)
        
        # Handle right join or full outer join
        if how in ['right', 'outer']:
            for right_i in range(other.row_count):
                if right_i not in matched_right_indices:
                    # Add left table nulls
                    for col in self.columns:
                        result_data[col].append(None)
                    # Add right DataFrame data
                    for col in other.columns:
                        if col != right_on:
                            result_data[col].append(other.data[col][right_i])
        
        return DataFrame(result_data)
    
    # ==================== Helper Methods ====================
    
    def head(self, n=5):
        """ return the first n rows of the DataFrame """
        new_data = {col: self.data[col][:n] for col in self.columns}
        return DataFrame(new_data)
    
    def tail(self, n=5):
        """ return the last n rows of the DataFrame """
        new_data = {col: self.data[col][-n:] for col in self.columns}
        return DataFrame(new_data)
    
    def shape(self):
        """ return (number of rows, number of columns) """
        return (self.row_count, len(self.columns))
    
    def info(self):
        """ display DataFrame information """
        print(f"DataFrame Info:")
        print(f"Rows: {self.row_count}")
        print(f"Columns: {len(self.columns)}")
        print(f"\nColumn Names and Types:")
        for col in self.columns:
            sample_val = self.data[col][0] if self.row_count > 0 else None
            val_type = type(sample_val).__name__
            print(f"  {col}: {val_type}")
    
    @classmethod
    def from_csv(cls, filepath, separator=","):
        """ create DataFrame from CSV file """
        data_dict = load_csv(filepath, separator)
        if data_dict is None:
            raise ValueError(f"Failed to load CSV from {filepath}")
        return cls(data_dict)


# ==================== GroupBy Class ====================

class GroupBy:
    def __init__(self, dataframe, by):
        """
        GroupBy object
        
        Parameters:
            dataframe: DataFrame
            by: list - columns to group by
        """
        self.df = dataframe
        self.by = by
        self.groups = self._create_groups()
    
    def _create_groups(self):
        """create group indices"""
        groups = {}
        
        for i in range(self.df.row_count):
            # create group key
            key_values = tuple(self.df.data[col][i] for col in self.by)
            
            if key_values not in groups:
                groups[key_values] = []
            groups[key_values].append(i)
        
        return groups
    
    def aggregate(self, agg_dict):
        """
        Aggregate operation
        
        Parameters:
            agg_dict: dict - {column_name: aggregation_function}
                Supported functions: 'sum', 'mean', 'max', 'min', 'count', 'std'
        
        Returns:
            DataFrame: aggregation result
        
        Example:
            df.groupby('Continent').aggregate({'GNP': 'max', 'Population': 'sum'})
        """
        result_data = {col: [] for col in self.by}
        
        # Create result columns for each aggregation column
        for col, func in agg_dict.items():
            result_data[f"{col}_{func}"] = []
        
        # Aggregate each group
        for key_values, indices in self.groups.items():
            # Add group keys
            for i, col in enumerate(self.by):
                result_data[col].append(key_values[i])
            
            # Calculate each aggregation column
            for col, func_name in agg_dict.items():
                values = [self.df.data[col][i] for i in indices]
                # Filter out None values
                values = [v for v in values if v is not None]
                
                if not values:
                    result = None
                else:
                    result = self._apply_aggregation(values, func_name)
                
                result_data[f"{col}_{func_name}"].append(result)
        
        return DataFrame(result_data)
    
    def _apply_aggregation(self, values, func_name):
        """Apply aggregation function"""
        if func_name == 'sum':
            return sum(values)
        elif func_name == 'mean':
            return sum(values) / len(values)
        elif func_name == 'max':
            return max(values)
        elif func_name == 'min':
            return min(values)
        elif func_name == 'count':
            return len(values)
        elif func_name == 'std':
            mean = sum(values) / len(values)
            variance = sum((x - mean) ** 2 for x in values) / len(values)
            return variance ** 0.5
        else:
            raise ValueError(f"Unknown aggregation function: {func_name}")
    
    def size(self):
        """Return the size of each group"""
        result_data = {col: [] for col in self.by}
        result_data['size'] = []
        
        for key_values, indices in self.groups.items():
                    for i, col in enumerate(self.by):
                        result_data[col].append(key_values[i])
                    result_data['size'].append(len(indices))

        
        return DataFrame(result_data)

In [9]:
# # test code
# if __name__ == "__main__":
#     # Create test CSV file
#     test_csv = """Name,Age,Country,GNP
# USA,250,United States,21000000
# China,70,People's Republic of China,14000000
# Japan,150,Japan,5000000
# Germany,100,Germany,4000000"""
    
#     with open("test_countries.csv", "w") as f:
#         f.write(test_csv)
    
#     # Method 1: Directly read as dictionary
#     data = load_csv("test_countries.csv")
#     print("Raw data dict:")
#     print(data)
#     print()
    
#     # Method 2: Using DataFrame class
#     df = DataFrame.from_csv("test_countries.csv")
#     print("DataFrame:")
#     print(df)
#     print()
    
#     # Test column access
#     print("Names column:")
#     print(df['Name'])
#     print()
    
#     # Test multiple column selection
#     print("Select Name and GNP:")
#     print(df[['Name', 'GNP']])
#     print()
    
#     # Test filtering (you need to implement condition checking)
#     print("Countries with GNP > 5000000:")
#     condition = [gnp > 5000000 for gnp in df['GNP']]
#     print(df[condition])

In [12]:
df = load_csv_advanced("../data/spotify_songs.csv", separator=",")
df_dataframe = DataFrame(df)
test_list = df_dataframe['track_popularity']

In [ ]:
# 1. loading data
countries = DataFrame.from_csv("countries.csv")
languages = DataFrame.from_csv("languages.csv")

# 2. filtering: find countries with GNP > 5 million
rich_countries = countries.filter(lambda row: row['GNP'] > 5000000)

# 3. selecting columns
result = rich_countries.select(['Name', 'GNP'])

# 4. grouping and aggregation: max GNP for each continent
continent_max = countries.groupby('Continent').aggregate({'GNP': 'max'})

# 5. joining: countries and languages
country_lang = countries.join(languages, 
                              left_on='Code', 
                              right_on='CountryCode')

# 6. chaining operations
result = (countries
    .filter(lambda row: row['Continent'] == 'Asia')
    .select(['Name', 'GNP'])
    .head(5))

In [31]:
df = load_csv_advanced("../data/spotify_songs.csv", separator=",")
df_dataframe = DataFrame(df)
df_dataframe.filter(lambda row: row['key'] > 5).head()

track_id             | track_name           | track_artist     | track_popularity | track_album_id       | track_album_name     | track_album_release_date | playlist_name | playlist_id          | playlist_genre | playlist_subgenre | danceability | energy | key | loudness | mode | speechiness | acousticness | instrumentalness | liveness | valence | tempo   | duration_ms
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
6f807x0ima9a1j3VP... | I Don't Care (wit... | Ed Sheeran       | 66               | 2oCs0DGTsRO98Gh5Z... | I Don't Care (wit... | 2019-06-14           | Pop Remix     | 37i9dQZF1DXcZDD7c... | pop            | dance pop         | 0.748        | 0.916  | 6  

--- 

# version 3
# completed function

In [ ]:
# ===================================================================
# 1. 核心輔助函數 (Core Helper Functions)
# ===================================================================

def _attempt_convert_type(value_str):
    """自動轉換數據類型"""
    # 處理空字符串
    if not value_str or value_str.strip() == '':
        return None
    
    value_str = value_str.strip()
    
    # 處理被引號包住的字符串
    if value_str.startswith('"') and value_str.endswith('"'):
        return value_str[1:-1]
    
    # 嘗試轉換為整數
    try:
        return int(value_str)
    except ValueError:
        pass
    
    # 嘗試轉換為浮點數
    try:
        return float(value_str)
    except ValueError:
        pass
    
    # 處理布林值
    if value_str.lower() in ['true', 'false']:
        return value_str.lower() == 'true'
    
    # 返回原始字符串
    return value_str

def parse_csv_line(line, separator=",", quote_char='"'):
    """
    解析一行 CSV, 正確處理引號內的分隔符
    """
    values = []
    current_value = ""
    in_quotes = False
    
    for char in line:
        if char == quote_char:
            in_quotes = not in_quotes
        elif char == separator and not in_quotes:
            values.append(current_value)
            current_value = ""
        else:
            current_value += char
    
    # 添加最後一個值
    values.append(current_value)
    return values


# ===================================================================
# 2. 核心 DataFrame 與 GroupBy 類 (Core DataFrame & GroupBy Classes)
# ===================================================================

class DataFrame:
    def __init__(self, data_dict):
        """
        使用列字典初始化 DataFrame
        data_dict: {'column1': [values], 'column2': [values]}
        """
        if not data_dict:
            raise ValueError("Cannot create DataFrame from empty data")
        
        # 檢查所有列是否長度一致
        lengths = [len(v) for v in data_dict.values()]
        if len(set(lengths)) > 1:
            # 找出長度不一致的列
            mismatched = {k: len(v) for k, v in data_dict.items()}
            raise ValueError(f"All columns must have the same length. Mismatched lengths: {mismatched}")
        
        self.data = data_dict
        self.columns = list(data_dict.keys())
        self.row_count = lengths[0] if lengths else 0
    
    def __repr__(self):
        """打印 DataFrame 的可讀表示"""
        if self.row_count == 0:
            return "Empty DataFrame"
        
        # 計算列寬
        col_widths = {}
        for col in self.columns:
            max_width = len(col)
            # 只檢查前 10 行以提高性能
            for val in self.data[col][:10]:
                max_width = max(max_width, len(str(val)))
            col_widths[col] = min(max_width, 20)  # 最大寬度限制為 20
        
        # 構建表格
        result = []
        
        # 標題行
        header = " | ".join(col.ljust(col_widths[col]) for col in self.columns)
        result.append(header)
        result.append("-" * len(header))
        
        # 數據行 (最多顯示 10 行)
        display_rows = min(10, self.row_count)
        for i in range(display_rows):
            row = []
            for col in self.columns:
                val = str(self.data[col][i])
                if len(val) > col_widths[col]:
                    val = val[:col_widths[col]-3] + "..."
                row.append(val.ljust(col_widths[col]))
            result.append(" | ".join(row))
        
        if self.row_count > 10:
            result.append(f"\n... {self.row_count - 10} more rows")
        
        result.append(f"\nShape: ({self.row_count} rows, {len(self.columns)} columns)")
        
        return "\n".join(result)
    
    def __getitem__(self, key):
        """
        實現 [] 索引操作
        - df['column'] (獲取單列)
        - df[['col1', 'col2']] (獲取多列)
        - df[boolean_list] (過濾)
        """
        if isinstance(key, str):
            # 返回單列 (作為一個 list)
            return self.data[key]
        elif isinstance(key, list) and all(isinstance(k, str) for k in key):
            # 返回多列 (作為一個新的 DataFrame)
            new_data = {col: self.data[col] for col in key}
            return DataFrame(new_data)
        elif isinstance(key, list) and all(isinstance(k, bool) for k in key):
            # 布林索引 (用於過濾)
            if len(key) != self.row_count:
                raise ValueError("Boolean index length mismatch")
            
            new_data = {col: [] for col in self.columns}
            for i, keep in enumerate(key):
                if keep:
                    for col in self.columns:
                        new_data[col].append(self.data[col][i])
            
            return DataFrame(new_data)
        else:
            raise TypeError(f"Invalid indexing type: {type(key)}")
    
    def __len__(self):
        """返回行數"""
        return self.row_count

    # ==================== 1. Filtering (篩選) ====================
    
    def filter(self, condition):
        """
        根據條件過濾行
        
        參數:
            condition: function (lambda) 或 list (布林值)
                - 函數: lambda row: row['Age'] > 18
                - 列表: [True, False, True, ...]
        
        返回:
            DataFrame: 過濾後的新 DataFrame
        """
        if callable(condition):
            # 函數條件
            keep_rows = []
            for i in range(self.row_count):
                row = {col: self.data[col][i] for col in self.columns}
                keep_rows.append(condition(row))
            return self[keep_rows]
        
        elif isinstance(condition, list) and all(isinstance(k, bool) for k in condition):
            # 布林列表
            return self[condition]
        
        else:
            raise TypeError("Condition must be a callable or list of booleans")
    
    # ==================== 2. Projection (投影) ====================
    
    def select(self, columns):
        """
        從 DataFrame 中選擇特定列
        
        參數:
            columns: str (單列) 或 list (多列)
        
        返回:
            DataFrame: 只包含所選列的新 DataFrame
        """
        if isinstance(columns, str):
            columns = [columns]
        
        return self[columns]
    
    # ==================== 3. GroupBy (分組) ====================
    
    def groupby(self, by):
        """
        按一列或多列分組
        
        參數:
            by: str 或 list - 用於分組的列名
        
        返回:
            GroupBy: GroupBy 對象
        """
        if isinstance(by, str):
            by = [by]
        
        return GroupBy(self, by)
    
    # ==================== 4. Join (連接) ====================
    
    def join(self, other, left_on, right_on, how='inner'):
        """
        與另一個 DataFrame 連接
        
        參數:
            other: DataFrame - 要連接的另一個 DataFrame
            left_on: str - 左側 DataFrame 的連接鍵
            right_on: str - 右側 DataFrame 的連接鍵
            how: str - 連接類型 ('inner', 'left', 'right', 'outer')
        
        返回:
            DataFrame: 連接後的新 DataFrame
        """
        # 為右側 DataFrame 創建索引
        right_index = {}
        for i, val in enumerate(other.data[right_on]):
            if val not in right_index:
                right_index[val] = []
            right_index[val].append(i)
        
        # 初始化結果字典
        result_data = {col: [] for col in self.columns}
        for col in other.columns:
            if col != right_on:  # 避免重複的鍵列
                result_data[f"{col}_right"] = [] # 重命名右側列以避免衝突
        
        matched_right_indices = set()
        
        # 遍歷左側 DataFrame
        for i in range(self.row_count):
            left_key = self.data[left_on][i]
            
            if left_key in right_index:
                # 找到匹配項
                for right_i in right_index[left_key]:
                    matched_right_indices.add(right_i)
                    
                    # 添加左側數據
                    for col in self.columns:
                        result_data[col].append(self.data[col][i])
                    
                    # 添加右側數據
                    for col in other.columns:
                        if col != right_on:
                            result_data[f"{col}_right"].append(other.data[col][right_i])
            elif how in ['left', 'outer']:
                # 左連接或全外連接: 保留左側行
                for col in self.columns:
                    result_data[col].append(self.data[col][i])
                for col in other.columns:
                    if col != right_on:
                        result_data[f"{col}_right"].append(None)
        
        # 處理右連接或全外連接
        if how in ['right', 'outer']:
            for right_i in range(other.row_count):
                if right_i not in matched_right_indices:
                    # 添加左側的空值
                    for col in self.columns:
                        result_data[col].append(None)
                    # 添加右側數據
                    for col in other.columns:
                        if col != right_on:
                            result_data[f"{col}_right"].append(other.data[col][right_i])
        
        return DataFrame(result_data)
    
    # ==================== 輔助方法 ====================
    
    def head(self, n=5):
        """返回前 n 行"""
        new_data = {col: self.data[col][:n] for col in self.columns}
        return DataFrame(new_data)
    
    def tail(self, n=5):
        """返回末尾 n 行"""
        new_data = {col: self.data[col][-n:] for col in self.columns}
        return DataFrame(new_data)
    
    def shape(self):
        """返回 (行數, 列數)"""
        return (self.row_count, len(self.columns))
    
    def info(self):
        """顯示 DataFrame 信息"""
        print(f"DataFrame Info:")
        print(f"Rows: {self.row_count}")
        print(f"Columns: {len(self.columns)}")
        print(f"\nColumn Names and Types:")
        for col in self.columns:
            sample_val = self.data[col][0] if self.row_count > 0 else None
            val_type = type(sample_val).__name__
            print(f"  {col}: {val_type}")
    
    @classmethod
    def from_csv(cls, filepath, separator=",", quote_char='"'):
        """從 CSV 文件創建 DataFrame (一次性加載)"""
        # *** 修復: 調用 load_csv_advanced ***
        data_dict = load_csv_advanced(filepath, separator, quote_char)
        if data_dict is None:
            raise ValueError(f"Failed to load CSV from {filepath}")
        return cls(data_dict)


# ==================== GroupBy Class (用於聚合) ====================

class GroupBy:
    def __init__(self, dataframe, by):
        """
        GroupBy 對象
        """
        self.df = dataframe
        self.by = by
        self.groups = self._create_groups()
    
    def _create_groups(self):
        """創建分組索引"""
        groups = {}
        
        for i in range(self.df.row_count):
            # 創建分組鍵 (元組)
            key_values = tuple(self.df.data[col][i] for col in self.by)
            
            if key_values not in groups:
                groups[key_values] = []
            groups[key_values].append(i)
        
        return groups
    
    def aggregate(self, agg_dict):
        """
        聚合操作
        
        參數:
            agg_dict: dict - {column_name: aggregation_function}
                支持的函數: 'sum', 'mean', 'max', 'min', 'count', 'std'
        
        返回:
            DataFrame: 聚合結果
        """
        result_data = {col: [] for col in self.by}
        
        # 為每個聚合創建結果列
        for col, func in agg_dict.items():
            result_data[f"{col}_{func}"] = []
        
        # 遍歷每個分組
        for key_values, indices in self.groups.items():
            # 添加分組鍵
            for i, col in enumerate(self.by):
                result_data[col].append(key_values[i])
            
            # 計算每個聚合
            for col, func_name in agg_dict.items():
                values = [self.df.data[col][i] for i in indices]
                # 過濾掉 None 值
                values = [v for v in values if v is not None]
                
                if not values:
                    result = None
                else:
                    result = self._apply_aggregation(values, func_name)
                
                result_data[f"{col}_{func_name}"].append(result)
        
        return DataFrame(result_data)
    
    def _apply_aggregation(self, values, func_name):
        """應用聚合函數"""
        if func_name == 'sum':
            return sum(values)
        elif func_name == 'mean':
            return sum(values) / len(values)
        elif func_name == 'max':
            return max(values)
        elif func_name == 'min':
            return min(values)
        elif func_name == 'count':
            return len(values)
        elif func_name == 'std':
            if len(values) < 1: # 避免除以零
                return None
            mean = sum(values) / len(values)
            variance = sum((x - mean) ** 2 for x in values) / len(values)
            return variance ** 0.5
        else:
            raise ValueError(f"Unknown aggregation function: {func_name}")
    
    def size(self):
        """返回每個分組的大小"""
        result_data = {col: [] for col in self.by}
        result_data['size'] = []
        
        for key_values, indices in self.groups.items():
            for i, col in enumerate(self.by):
                result_data[col].append(key_values[i])
            result_data['size'].append(len(indices))

        return DataFrame(result_data)
  

# ===================================================================
# 3. CSV 解析器 (加載至內存) (In-Memory CSV Parser)
# ===================================================================

def load_csv_advanced(filePath, separator=",", quote_char='"'):
    """
    高級 CSV 加載器, 處理引號內的字段並加載到內存
    (例如: "Smith, John",25,USA)
    """
    
    # *** 注意: 這裡不再定義內部的 parse_csv_line ***
    # *** 我們將使用全局的 parse_csv_line 函數 ***
    
    data_dict = {}
    header = []
    
    try:
        with open(filePath, "r", encoding="utf-8") as f:
            # 讀取標題行
            header_line = f.readline().strip()
            # 使用全局的解析函數
            header = parse_csv_line(header_line, separator, quote_char)
            header = [col.strip().strip(quote_char) for col in header] # 額外清理
            
            data_dict = {col_name: [] for col_name in header}
            
            # 讀取數據行
            for line_number, line in enumerate(f, start=2):
                cleaned_line = line.strip()
                if not cleaned_line:
                    continue
                
                # 使用全局的解析函數
                values = parse_csv_line(cleaned_line, separator, quote_char)
                
                if len(values) == len(header):
                    for i, col_name in enumerate(header):
                        converted_value = _attempt_convert_type(values[i])
                        data_dict[col_name].append(converted_value)
                else:
                    print(f"Warning: Line {line_number} column mismatch (Expected {len(header)}, got {len(values)})")
        
        return data_dict
        
    except FileNotFoundError:
        print(f"Error: File not found at {filePath}")
        return None
    except Exception as e:
        print(f"Error reading CSV: {e}")
        import traceback
        traceback.print_exc()
        return None


# ===================================================================
# 4. 分批處理 (Chunked/Batch Processing)
# ===================================================================

class CSVChunkReader:
    """
    分塊讀取 CSV 文件
    使用生成器模式, 每次返回指定數量的行 (作為一個 DataFrame)
    適用於處理大型 CSV 文件
    """
    
    def __init__(self, filepath, separator=",", quote_char='"', chunk_size=1000):
        """
        初始化分塊讀取器
        
        參數:
            filepath: str - CSV 文件路徑
            separator: str - 分隔符
            quote_char: str - 引號字符 (*** 新增 ***)
            chunk_size: int - 每塊的行數
        """
        self.filepath = filepath
        self.separator = separator
        self.quote_char = quote_char # *** 新增 ***
        self.chunk_size = chunk_size
        self.header = None
        self.total_rows_read = 0
    
    def __iter__(self):
        """返回迭代器"""
        return self.read_chunks()
    
    def read_chunks(self):
        """
        生成器函數, 逐塊返回 DataFrame
        
        Yields:
            DataFrame - 每次返回 chunk_size 行的 DataFrame
        """
        try:
            with open(self.filepath, 'r', encoding='utf-8') as f:
                # 讀取標題行
                header_line = f.readline().strip()
                if not header_line:
                    raise ValueError("File is empty")
                
                # *** 修改: 傳遞 quote_char ***
                self.header = parse_csv_line(header_line, self.separator, self.quote_char)
                self.header = [col.strip().strip(self.quote_char) for col in self.header]
                
                chunk_data = {col: [] for col in self.header}
                chunk_row_count = 0
                
                # 逐行讀取
                for line_number, line in enumerate(f, start=2):
                    cleaned_line = line.strip()
                    
                    if not cleaned_line:
                        continue
                    
                    # *** 修改: 傳遞 quote_char ***
                    values = parse_csv_line(cleaned_line, self.separator, self.quote_char)
                    
                    if len(values) == len(self.header):
                        # 添加到當前塊
                        for i, col in enumerate(self.header):
                            converted_value = _attempt_convert_type(values[i])
                            chunk_data[col].append(converted_value)
                        
                        chunk_row_count += 1
                        self.total_rows_read += 1
                        
                        # 如果達到塊大小, 返回這個塊
                        if chunk_row_count >= self.chunk_size:
                            yield DataFrame(chunk_data)
                            # 重置塊數據
                            chunk_data = {col: [] for col in self.header}
                            chunk_row_count = 0
                    else:
                        print(f"Warning: Line {line_number} column mismatch (Expected {len(self.header)}, got {len(values)})")
                
                # 返回最後一個不完整的塊
                if chunk_row_count > 0:
                    yield DataFrame(chunk_data)
        
        except FileNotFoundError:
            print(f"Error: File not found at {self.filepath}")
            return
        except Exception as e:
            print(f"Error reading CSV chunks: {e}")
            import traceback
            traceback.print_exc()
            return


def read_csv_chunks(filepath, separator=",", quote_char='"', chunk_size=1000):
    """
    便捷函數: 分塊讀取 CSV
    
    示例:
        for chunk in read_csv_chunks("large_file.csv", chunk_size=1000):
            # 處理每個塊
            print(chunk.shape())
    """
    # *** 修改: 傳遞 quote_char ***
    return CSVChunkReader(filepath, separator, quote_char, chunk_size)


class ChunkProcessor:
    """
    分塊處理器 - 提供常用的分塊處理操作
    (注意: 這裡的分組聚合是簡化版, 對於複雜聚合可能需要更高級的合併策略)
    """
    
    @staticmethod
    def filter_chunks(chunk_reader, condition):
        """
        對每個塊應用過濾
        """
        for chunk in chunk_reader:
            filtered = chunk.filter(condition)
            if len(filtered) > 0:
                yield filtered
    
    @staticmethod
    def aggregate_chunks(chunk_reader, group_by_cols, agg_dict):
        """
        對所有塊進行聚合 (簡化版)
        """
        if isinstance(group_by_cols, str):
            group_by_cols = [group_by_cols]
        
        # 累積的分組數據
        accumulated_groups = {}
        
        for chunk in chunk_reader:
            # 對當前塊進行分組
            grouped = chunk.groupby(group_by_cols)
            chunk_result = grouped.aggregate(agg_dict)
            
            # 合併到累積結果
            for i in range(len(chunk_result)):
                # 創建組鍵
                group_key = tuple(chunk_result.data[col][i] for col in group_by_cols)
                
                if group_key not in accumulated_groups:
                    # 初始化該組
                    accumulated_groups[group_key] = {}
                    for col in group_by_cols:
                        accumulated_groups[group_key][col] = chunk_result.data[col][i]
                    # 為聚合列初始化列表
                    for agg_col in agg_dict.keys():
                        func = agg_dict[agg_col]
                        result_col = f"{agg_col}_{func}"
                        accumulated_groups[group_key][result_col] = []
                
                # 累積聚合值 (這裡只是簡單收集, 最後再計算)
                for agg_col in agg_dict.keys():
                    func = agg_dict[agg_col]
                    result_col = f"{agg_col}_{func}"
                    accumulated_groups[group_key][result_col].append(
                        chunk_result.data[result_col][i]
                    )
        
        # 最終聚合
        final_data = {col: [] for col in group_by_cols}
        for agg_col in agg_dict.keys():
            func = agg_dict[agg_col]
            final_data[f"{agg_col}_{func}"] = []
        
        for group_key, group_data in accumulated_groups.items():
            # 添加分組鍵
            for i, col in enumerate(group_by_cols):
                final_data[col].append(group_key[i])
            
            # 最終聚合計算
            for agg_col in agg_dict.keys():
                func = agg_dict[agg_col]
                result_col = f"{agg_col}_{func}"
                values = [v for v in group_data[result_col] if v is not None]
                
                if not values:
                    final_value = None
                elif func == 'sum':
                    final_value = sum(values)
                elif func == 'max':
                    final_value = max(values)
                elif func == 'min':
                    final_value = min(values)
                elif func == 'count':
                    final_value = sum(values) # 假設 'count' 是預先計算好的
                elif func == 'mean':
                    # 注意: 均值的均值是不對的, 這裡假設傳入的是'sum'和'count'
                    # 為了簡化, 這裡我們只對收集到的'值'求均值
                    final_value = sum(values) / len(values) 
                else:
                    final_value = values[0] # 其他情況取第一個
                
                final_data[result_col].append(final_value)
        
        return DataFrame(final_data)
    
    @staticmethod
    def count_chunks(chunk_reader):
        """
        計算所有塊的總行數
        """
        total = 0
        for chunk in chunk_reader:
            total += len(chunk)
        return total
    
    @staticmethod
    def collect_chunks(chunk_reader, max_rows=None):
        """
        將所有塊合併成一個 DataFrame
        (注意: 只有當數據能裝入內存時才使用)
        """
        all_data = None
        total_rows = 0
        
        for chunk in chunk_reader:
            if all_data is None:
                all_data = {col: [] for col in chunk.columns}
            
            if max_rows and total_rows >= max_rows:
                break

            rows_to_add = len(chunk)
            if max_rows and total_rows + rows_to_add > max_rows:
                rows_to_add = max_rows - total_rows

            for col in chunk.columns:
                all_data[col].extend(chunk.data[col][:rows_to_add])
            
            total_rows += rows_to_add
        
        if all_data:
            return DataFrame(all_data)
        return DataFrame({}) # 返回空的 DataFrame